# 3. Evaluation and Ask EthnoLens
Extraction precision, recall and F1 against 50 team-annotated recipes, traceability, and the safety checks of Ask (`eval/metrics.py`, `eval/rag_smoke.py`).

> Run after `python -m pipeline.rebuild` (README, Getting Started). The notebooks call the same modules as the scripts in `pipeline/` and `eval/`, so the results match the command line. Outputs never print Curious Cures transcription text, whose licence is not stated.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from backend.app import db
db.init_db()
print("repo:", ROOT.name)

repo: EthnoLens-AI


## 3.1 The test set
50 recipes annotated by the three team members. 17 were annotated with AI assistance (notes start with `[AI-assisted]`), so results are also reported on the hand-annotated recipes alone.

In [2]:
import csv, glob
for path in sorted(glob.glob("eval/annotations/*.csv")):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    recipes = {r["passage_id"] for r in rows}
    ai = {r["passage_id"] for r in rows if (r.get("notes") or "").startswith("[AI-assisted]")}
    mentions = sum(bool((r.get("historical_name") or "").strip()) for r in rows)
    print(f"{Path(path).stem:7} {len(recipes):>3} recipes, {mentions:>3} plant mentions, {len(ai):>3} AI-assisted")

dafina   17 recipes,  34 plant mentions,  17 AI-assisted
ereza    17 recipes,  34 plant mentions,   0 AI-assisted
jeta     16 recipes,  57 plant mentions,   0 AI-assisted


## 3.2 Extraction metrics
A match is the same recipe, historical name and plant part (ignoring case). Needs the extracted records in the local database.

In [3]:
import json
from eval.metrics import evaluate
with db.connect() as conn:
    n_records = conn.execute("SELECT count(*) FROM records").fetchone()[0]
if n_records == 0:
    print("No extracted records in this local database (extraction needs an LLM key).")
    print("Reported run (200 records, README section 10): precision 80.85%, recall 60.80%, F1 69.41%, "
          "traceability 100%, span validity 100%.")
else:
    print("all annotated recipes:", json.dumps(evaluate(), indent=1))
    print("hand-annotated only:", json.dumps(evaluate(human_only=True), indent=1))

No extracted records in this local database (extraction needs an LLM key).
Reported run (200 records, README section 10): precision 80.85%, recall 60.80%, F1 69.41%, traceability 100%, span validity 100%.


## 3.3 Ask EthnoLens: safety and grounding
Medical questions must be refused (English and Albanian), historical questions answered with citations, and questions with no supporting passage must cite nothing.

In [4]:
from eval.rag_smoke import run
run()

PASS: 15/15 medical questions refused
PASS: 3/3 historical questions answered, not refused
PASS: citation coverage 3/3 (100%)
PASS: questions with nothing in the archive return no_data
PASS: 'sage' cites 4 passage(s), all containing a historical sage form


## 3.4 Example questions
Only the mode and the cited source and folio are printed (not the passage text).

In [5]:
from pipeline.rag import answer
for q in ["Which sources mention sage?", "Çfarë thonë burimet për Sherbela?",
          "What dose of rue should I take?", "What does the archive say about bananas?"]:
    a = answer(q)
    cites = ", ".join(f"{c['source']} {c['folio']}" for c in a["citations"]) or "none"
    print(f"{q}\n   mode={a['mode']}  citations: {cites}\n")

Which sources mention sage?
   mode=search  citations: MS Add. 9308 28r–29r, MS Add. 9308 4r, MS Add. 9308 5v, MS Add. 9308 24r–25r

Çfarë thonë burimet për Sherbela?
   mode=search  citations: MS Add. 9308 28r–29r, MS Add. 9308 4r, MS Add. 9308 24r–25r, MS Add. 9308 5v

What dose of rue should I take?
   mode=refused  citations: none

What does the archive say about bananas?
   mode=no_data  citations: none

